# Practical 3: Building a RAG over Digital-Forensics Reports

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danjethh/sectratek-AI-Lab/blob/main/df_llm_labs/lab3_rag/03_lab_notebook.ipynb)

**Course:** AI for Cybersecurity and Digital Forensics  
**Instructor:** Daniel Kwaku Ntiamoah Addai, Mobile Threat Research Lead, SectraTEK Lab  
**Runs in:** Google Colab or any hosted Jupyter. Nothing is installed on your own computer.

---

An examiner builds up many case reports over time. **RAG** (Retrieval-Augmented Generation) lets you
ask plain-English questions over that pile of reports and get an answer that is **grounded in the
reports and cites them**, instead of a confident guess from the model's memory.

In this lab you build a small RAG system **by hand**, one step at a time:

1. **Chunk** the reports into short passages.
2. **Embed** each passage as a vector of numbers that captures its meaning.
3. **Retrieve** the passages closest in meaning to a question.
4. **Generate** an answer from only those passages, with citations.

The forensic point is the same one that runs through this whole course: the answer must stay inside
what the evidence (here, the reports) actually says.

## Part 0: One-Time Setup

Run these cells in order. They install three libraries, connect you to a **free hosted model**
(Groq) for the writing step, and save the case reports next to the notebook. No local install is
required; everything happens inside this hosted session.

### Step 0.1: Install the libraries

**Purpose:** install the three tools this lab uses.  
**What you'll see:** a pip log that takes a minute (it downloads a small embedding model library).
Warnings are normal.

In [ ]:
# Purpose: install the one-time libraries this notebook needs, inside the hosted session only.
# sentence-transformers: turns text into 'embeddings' (vectors of numbers that capture meaning)
# numpy: fast math on those vectors (we use it to measure how close two meanings are)
# openai: the client that sends the final question to the model (Groq uses the same interface)
%pip install -q sentence-transformers numpy openai

### Step 0.2: Connect to a free hosted model (Groq)

**Purpose:** get a free **API key** (a private password that lets the notebook use the model) so the
last step can write an answer in words. The retrieval steps before it do not need the model.

1. Open **https://console.groq.com/keys**, sign in, and create a key.
2. Run the cell and paste it when asked. It is hidden and is **not saved** into the notebook.

**What you'll see:** `Model OK: ...` once the connection works.

In [ ]:
# Purpose: collect the Groq key privately and build a client for the answer-writing step.
import os        # os: lets the notebook read saved settings, such as where your key is kept
import getpass   # getpass: lets you type your secret key without it appearing on screen
from openai import OpenAI  # OpenAI: the client that carries your question to the model (Groq speaks the same language)

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass('Paste your Groq API key (hidden): ').strip()

BASE_URL = 'https://api.groq.com/openai/v1'
MODEL = 'llama-3.3-70b-versatile'  # free and strong. If this name ever errors, run the model-list cell.

client = OpenAI(base_url=BASE_URL, api_key=os.environ['GROQ_API_KEY'])

try:
    probe = client.chat.completions.create(
        model=MODEL, messages=[{'role': 'user', 'content': 'Reply with one word: ready'}])
    print('Model OK:', MODEL, '->', probe.choices[0].message.content.strip())
except Exception as err:
    print('Could not use', MODEL, '->', err)
    print('\nModels available on your key:')
    try:
        for m in client.models.list().data:
            print('  -', m.id)
    except Exception as err2:
        print('  (could not list models:', err2, ')')
    print('\nSet MODEL above to one of these and re-run.')

### Step 0.3: Save the case reports

**Purpose:** write six short, synthetic digital-forensics reports into a `reports/` folder. These are
the knowledge base your RAG will search.  
**What you'll see:** the list of report files written.

In [ ]:
# Purpose: create the report files on disk so the notebook has its own knowledge base, with no download.
from pathlib import Path  # Path: a readable way to refer to files and folders by name

REPORTS = {'DF-2026-017_mobile_exfiltration.md': '# Case DF-2026-017: Suspected Customer-Data Exfiltration\n\nExaminer: Analyst B, SectraTEK Lab\nDevice: Google Pixel 7 (Android 14)\nAcquisition: full file system, Cellebrite UFED 7.64\nIncident window (UTC): 2026-02-15 01:00 to 01:15\n\n## Summary\nA company-issued Android phone was reviewed after a possible leak of the file\ncustomers_q1.csv before an employee\'s departure. The artifacts show the file being staged in\nTelegram storage, outbound Telegram and Gmail messages, related network activity, and deletion\nof the original file from Downloads.\n\n## Key findings\n- 01:05:22 - customers_q1.csv created in /Download.\n- 01:07:41 - the same file hash copied into Telegram\'s Documents folder.\n- 01:07:58 - outbound Telegram message: "sending that sheet now".\n- 01:09:44 - outbound Gmail message: "see attached export".\n- 01:12:04 - customers_q1.csv deleted from Downloads.\n\n## Indicators\n- SHA-256 of customers_q1.csv: 531ec9ebf8f4492754f16b1437f1824995c8bf4ee2f6be74d4f2f13fdf3f2ab5\n- Network endpoints: api.telegram.org, smtp.gmail.com\n\n## Conclusion\nThe evidence supports staging and attempted external sharing of the file through Telegram and\nGmail. It does not, on its own, confirm that the file was delivered to or received by any\nrecipient. Recommend correlation with recipient-side or provider records.\n', 'DF-2026-021_windows_ransomware.md': '# Case DF-2026-021: Windows Ransomware Incident\n\nExaminer: Analyst C, SectraTEK Lab\nDevice: Dell OptiPlex workstation (Windows 11)\nAcquisition: E01 disk image, FTK Imager\n\n## Summary\nA finance workstation was encrypted by ransomware. Hundreds of documents were renamed with a\n.lxk extension and a ransom note, README_RECOVER.txt, was dropped in every affected folder.\n\n## Key findings\n- Initial access via a malicious macro in an email attachment, invoice_8841.xlsm.\n- The macro downloaded a loader from a staging host and executed it from %TEMP%.\n- Volume Shadow Copies were deleted with vssadmin to prevent recovery.\n- Encryption began 2026-03-04 22:11 UTC and ran for roughly 40 minutes.\n\n## Indicators\n- Ransom note filename: README_RECOVER.txt\n- Encrypted extension: .lxk\n- Bitcoin address in note: 1FakeRansomAddr00000000000000000000\n- Loader SHA-256: 9a1d4c0b7e2f5a6c8d3b1e0f4a7c9d2b6e5f8a1c3d7b2e9f0a4c6d8b1e3f5a7c\n- Staging host: updates.fake-cdn-host.net\n\n## Conclusion\nThe incident is consistent with a macro-delivered ransomware loader that disabled local\nrecovery before encrypting user documents. Behavior resembles common LockBit-style tooling, but\nattribution is not confirmed. No evidence of data exfiltration was found in the available logs.\n', 'DF-2026-025_memory_credential_theft.md': '# Case DF-2026-025: Memory Forensics and Credential Theft\n\nExaminer: Analyst B, SectraTEK Lab\nDevice: HP EliteBook laptop (Windows 10)\nAcquisition: live RAM capture (16 GB), analyzed with Volatility 3\n\n## Summary\nA laptop suspected of credential theft was captured while powered on. Memory analysis found a\ncredential-dumping tool resident in memory and signs that domain credentials were harvested.\n\n## Key findings\n- A process named svch0st.exe (note the zero) was running from a user profile, not System32.\n- Strings consistent with Mimikatz (sekurlsa, logonpasswords) were present in that process.\n- The LSASS process had an unusual handle opened by svch0st.exe, consistent with a memory read.\n- A scheduled task re-launched the tool at logon for persistence.\n\n## Indicators\n- Malicious process: svch0st.exe (PID 4821)\n- Parent process: powershell.exe launched from a macro\n- Persistence: scheduled task "WindowsUpdateHelper"\n- Observed technique: LSASS memory access for credential extraction\n\n## Conclusion\nThe evidence supports in-memory credential dumping of domain accounts via a Mimikatz-like tool.\nHarvested credentials may enable lateral movement. Recommend a domain-wide password reset and\nreview of authentication logs for pass-the-hash activity.\n', 'DF-2026-030_ios_stalkerware.md': '# Case DF-2026-030: iOS Stalkerware Assessment\n\nExaminer: Analyst A, SectraTEK Lab\nDevice: Apple iPhone 13 (iOS 17)\nAcquisition: logical + sysdiagnose, analyzed with MVT and manual review\n\n## Summary\nA personal iPhone was examined after the owner reported being tracked. The device showed signs\nconsistent with a commercial monitoring app (stalkerware), including a configuration profile and\nperiodic background network beaconing.\n\n## Key findings\n- A configuration profile granted VPN and web-filter permissions, installed outside MDM.\n- Periodic HTTPS beacons every 15 minutes to a monitoring back-end during charging hours.\n- Elevated background location and microphone access by a non-App-Store helper process.\n- The app icon was hidden from the home screen.\n\n## Indicators\n- Beacon domain: sync.fake-familytrack.app\n- Beacon interval: approximately every 15 minutes\n- Profile display name: "Device Services"\n- Classification: dual-use monitoring / stalkerware (web-verified vendor)\n\n## Conclusion\nFindings are consistent with installed stalkerware performing location and audio monitoring. On\nan unsupervised device, profile payload contents are limited, so some detections are\ncapability-dependent. Recommend removing the profile and rotating the owner\'s account passwords.\n', 'DF-2026-034_disk_deleted_recovery.md': '# Case DF-2026-034: Disk Image and Deleted-File Recovery\n\nExaminer: Analyst C, SectraTEK Lab\nDevice: external USB drive (exFAT), 128 GB\nAcquisition: raw dd image, verified by SHA-256\n\n## Summary\nAn external drive suspected of holding deleted evidence was imaged and carved. Several images\nand a spreadsheet were recovered from unallocated space, and signs of anti-forensic wiping were\nobserved.\n\n## Key findings\n- File carving recovered 42 JPEG images and one XLSX from unallocated clusters.\n- Filesystem metadata for the originals had been overwritten, but file content survived.\n- A wiping utility had been run, zeroing the directory entries but not all data clusters.\n- Timestamps on recovered content predated the reported wipe by several days.\n\n## Indicators\n- Recovered artifacts: 42 x JPEG, 1 x XLSX (from unallocated space)\n- Anti-forensic tool signature: bulk directory-entry zeroing\n- Verification: image SHA-256 matched the acquisition log\n\n## Conclusion\nThe evidence supports an attempt to destroy files by wiping directory entries, which did not\nfully remove the underlying data. Carving recovered usable content. The recovered spreadsheet\nshould be compared by hash against known case files.\n', 'DF-2026-039_network_intrusion.md': "# Case DF-2026-039: Network Intrusion and Lateral Movement\n\nExaminer: Analyst A, SectraTEK Lab\nScope: corporate network, endpoint and PCAP review\nAcquisition: endpoint logs plus full packet capture (Zeek + Suricata)\n\n## Summary\nAn intrusion began with a phishing email and progressed to lateral movement and command-and-\ncontrol over DNS. Two servers were accessed before containment.\n\n## Key findings\n- Initial access: a phishing link harvested a user's credentials on a lookalike login page.\n- The attacker used valid credentials over VPN during off-hours.\n- Lateral movement used SMB and remote service creation between workstations.\n- Command-and-control used DNS tunnelling with long, encoded subdomains.\n\n## Indicators\n- Phishing domain: login.fake-sso-portal.com\n- C2 technique: DNS tunnelling (encoded TXT queries)\n- C2 domain: c2.fake-telemetry-dns.net\n- Lateral movement: SMB plus remote service creation (technique T1021)\n\n## Conclusion\nThe evidence supports a credential-phishing intrusion followed by hands-on-keyboard lateral\nmovement and DNS-based C2. No ransomware or encryption was observed. Recommend credential\nresets, DNS egress monitoring, and review of the two accessed servers for data staging.\n"}

reports_dir = Path('reports'); reports_dir.mkdir(exist_ok=True)
for name, text in REPORTS.items():
    (reports_dir / name).write_text(text)
print('Saved', len(REPORTS), 'reports to', reports_dir.resolve())
for name in REPORTS:
    print('  -', name)

## Part A: Build the RAG by Hand

You will do the four RAG steps one at a time, so each is visible before you ask any questions.

### Step A.1: Load the reports and split them into passages (chunking)

**Purpose:** a whole report is too long to compare against a short question, so we cut each report
into short **passages** (chunks). Each chunk remembers which report it came from, so we can cite it
later.  
**What you'll see:** how many chunks we made, and one example chunk.

In [ ]:
# Purpose: read every report and split it into short, labelled passages.
from pathlib import Path

reports_dir = Path('reports')

def split_into_chunks(text, max_words=90):
    # Split on blank lines (paragraphs), then group paragraphs until a chunk is about max_words long.
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    chunks, current = [], ''
    for p in paragraphs:
        if len((current + ' ' + p).split()) > max_words and current:
            chunks.append(current.strip()); current = p
        else:
            current = (current + '\n\n' + p).strip()
    if current:
        chunks.append(current.strip())
    return chunks

# Build a list of passages; each one keeps its source report name for citations.
passages = []
for path in sorted(reports_dir.glob('*.md')):
    for chunk in split_into_chunks(path.read_text()):
        passages.append({'source': path.name, 'text': chunk})

print('Made', len(passages), 'passages from', len(list(reports_dir.glob('*.md'))), 'reports.')
print('\nExample passage (from', passages[1]['source'] + '):\n')
print(passages[1]['text'][:300], '...')

### Step A.2: Turn each passage into an embedding (a vector of numbers)

**Purpose:** a computer cannot compare meanings directly, so we convert each passage into an
**embedding**: a list of numbers where passages with similar meaning end up with similar numbers. We
use a small, free model (`all-MiniLM-L6-v2`) that runs right here in the session.  
**What you'll see:** the shape of the embedding table, for example `(N passages, 384 numbers each)`.

In [ ]:
# Purpose: load a small embedding model and turn every passage into a vector.
from sentence_transformers import SentenceTransformer  # SentenceTransformer: converts text into meaning-vectors
import numpy as np  # numpy: lets us store and compare those vectors efficiently

# Download the small model once (about 90 MB). It runs on the free CPU in Colab.
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Embed every passage. normalize_embeddings makes the closeness score in the next step simple to read.
passage_vectors = embedder.encode([p['text'] for p in passages], normalize_embeddings=True)
passage_vectors = np.array(passage_vectors)
print('Embedding table shape:', passage_vectors.shape, '(passages, numbers per passage)')

### Step A.3: Retrieve the passages closest to a question

**Purpose:** to answer a question, we embed the **question** the same way, then find the passages
whose vectors are closest to it. Closeness is measured by a **cosine similarity** score from 0 to 1;
higher means more similar in meaning.  
**What you'll see:** the top passages for an example question, with their scores and sources.

In [ ]:
# Purpose: a search function that returns the passages most related to a question.
def retrieve(question, k=4):
    q_vector = embedder.encode([question], normalize_embeddings=True)[0]
    # Because vectors are normalized, the dot product IS the cosine-similarity score (0 to 1).
    scores = passage_vectors @ q_vector
    top = np.argsort(scores)[::-1][:k]   # the indexes of the k highest scores
    return [{'score': float(scores[i]), 'source': passages[i]['source'], 'text': passages[i]['text']} for i in top]

example_q = 'Which cases involved stolen or dumped credentials?'
for hit in retrieve(example_q, k=3):
    print(f"[{hit['score']:.2f}] {hit['source']}")
    print('   ', hit['text'][:120].replace('\n', ' '), '...\n')

### Step A.4: Generate a grounded, cited answer

**Purpose:** now we give the model **only the retrieved passages** and the question, and ask it to
answer from those passages and cite the report names. If the answer is not in the passages, it must
say so, instead of guessing.  
**What you'll see:** a short answer that cites the reports it used.

In [ ]:
# Purpose: build a prompt from the retrieved passages and ask the model to answer with citations.
def answer(question, k=4):
    hits = retrieve(question, k=k)
    context = '\n\n'.join(f"[{h['source']}]\n{h['text']}" for h in hits)
    prompt = (
        'You are a digital-forensics assistant. Answer the question using ONLY the report excerpts '
        'below. Cite the report file name in square brackets after each claim. If the answer is not in '
        'the excerpts, say "That is not covered in the reports." Do not use outside knowledge.\n\n'
        f'Report excerpts:\n{context}\n\nQuestion: {question}')
    reply = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
    ).choices[0].message.content
    return reply, hits

reply, hits = answer('Which cases involved stolen or dumped credentials, and how?')
print(reply)
print('\n--- sources retrieved ---')
for h in hits:
    print(f"  [{h['score']:.2f}] {h['source']}")

## Part B: Why Retrieval Matters

To see what RAG adds, ask the **same question two ways**: once with the model alone (no reports), and
once with retrieval. Watch how the plain model is happy to guess specifics, while the RAG answer stays
inside the reports and cites them.

### Step B.1: Ask the model with no reports

**Purpose:** a question about *our* cases that the model cannot know. Without retrieval it will either
refuse or invent an answer.  
**What you'll see:** a generic or made-up answer, with no citations.

In [ ]:
# Purpose: the same question, but the model gets no reports to ground it.
bare_question = 'In case DF-2026-025, what tool was used to steal credentials, and how did it persist?'
bare = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': bare_question}],
).choices[0].message.content
print(bare)

### Step B.2: Ask the same question with retrieval

**Purpose:** now the RAG pipeline finds the right report and answers from it, with a citation.  
**What you'll see:** a specific answer grounded in `DF-2026-025`, naming the tool and the persistence.

In [ ]:
# Purpose: the same question, answered by the RAG pipeline you built.
reply, hits = answer('In case DF-2026-025, what tool was used to steal credentials, and how did it persist?')
print(reply)
print('\n--- sources retrieved ---')
for h in hits:
    print(f"  [{h['score']:.2f}] {h['source']}")

## Part C: Ask Your Own Questions

Use the `answer(...)` function to query the whole report set. Try cross-report questions, the kind a
lead examiner would ask across many cases.

In [ ]:
# Purpose: your turn. Change the question and re-run.
my_question = 'Which cases showed anti-forensic behaviour, such as deleting backups or wiping files?'
reply, hits = answer(my_question)
print(reply)
print('\n--- sources retrieved ---')
for h in hits:
    print(f"  [{h['score']:.2f}] {h['source']}")

### Good questions to try

- *Which cases involved Android devices?*
- *Summarise every case that mentions ransomware or encryption.*
- *What network indicators (domains or addresses) appear across the reports?*
- *Which cases did Analyst B work on?*
- *Was customer data confirmed delivered in the exfiltration case?* (watch it stay bounded)

## Your Short Write-Up

Answer in the cell below, in your own words.

1. Pick one question. Paste the RAG answer, and confirm the cited report actually supports it.
2. Find a question the reports **cannot** answer. Did the system say so, or did it guess?
3. In one sentence: why does retrieval make the answer more trustworthy than the plain model?

In [ ]:
writeup = {
    'q1_grounded_answer': '... your question, the answer, and the citation you checked ...',
    'q2_out_of_scope': '... a question not in the reports, and what the system did ...',
    'q3_why_retrieval_helps': '... one sentence ...',
}
for k, v in writeup.items():
    print(k, '->', v)

## Submit

Save the notebook with the retrieval output, the with/without comparison, and your write-up filled in.

- In Colab: **File -> Download -> Download .ipynb**, or **File -> Save a copy in Drive**.

---

*The reports in this lab are synthetic, written for teaching. The RAG pipeline (chunk, embed, retrieve,
generate) is built from scratch with `sentence-transformers` and `numpy`, with Groq for the final
answer, so you can see every step.*